In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Install libraries

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U accelerate
!pip install -q -U trl
!pip install -q -U transformers peft datasets

In [ ]:
import os
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GemmaTokenizer
from trl import SFTTrainer

# Load the data

In [ ]:
import pandas as pd
import numpy as np


train_data = pd.read_csv("/kaggle/input/emoti-code-multi-script-emotion-classification-rel/competition_train.csv")
val_data = pd.read_csv("/kaggle/input/emoti-code-multi-script-emotion-classification-rel/competition_val.csv")
test_data = pd.read_csv("/kaggle/input/emoti-code-multi-script-emotion-classification-rel/competition_test.csv")

In [ ]:
train_data

In [ ]:
test_data

In [ ]:
train_data.emotion.unique()

In [ ]:
val_data.emotion.unique()

# Load the token

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("DLP_NPPE_1")

In [ ]:
from huggingface_hub import login

login(secret_value_0)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Fetch token from Kaggle secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("DLP_NPPE_1")

# Log in to Hugging Face
login(hf_token)

# Load the model and fine tune using QLoRA and LoRA

In [ ]:
model_id = "google/gemma-3-1b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map = {"":0},
    token = hf_token)

In [ ]:
os.environ['WANDB_DISABLED'] = "false"

In [ ]:
text = "review: The food is below average,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
lora_config =  LoraConfig(
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_data)
val_ds = Dataset.from_pandas(val_data)
test_ds = Dataset.from_pandas(test_data)

print(train_ds, val_ds, test_ds)

In [ ]:
def formatting_func(example):
    text = (
        f"Sentence: {example['Sentence']}, Language: {example['language']}\n"
        f"What is the emotion expressed in this sentence?\n"
        f"Only respond with one word, choosing exactly one of [disgust, anger, sad, happy, fear, surprise].\n"
        f"Answer in lowercase letters only: {example['emotion'].lower()}"
    )
    return text



In [ ]:
from datasets import concatenate_datasets

full_ds = concatenate_datasets([train_ds, val_ds])

print(full_ds)

# Supervised Fine-tuning of the model

In [ ]:
from transformers import TrainingArguments

trainer =SFTTrainer(
    model = model,
    train_dataset = full_ds,
    args = TrainingArguments(
        output_dir="./gemma-qlora-emotion",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        logging_steps=50,
        fp16 = True,
        optim = "paged_adamw_8bit",
        num_train_epochs=5,
        learning_rate=2e-4,
        report_to="none"
        ),
    peft_config = lora_config ,
    formatting_func=formatting_func
)


In [ ]:
trainer.train()

# Evaluating f1 score on the Validation Dataset

In [ ]:
import pandas as pd
import torch
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
from torch.cuda.amp import autocast

device = "cuda:0"
model = model.to(device)

pred_emotions = []

for i, row in tqdm(val_data.iterrows(), total=len(val_data)):
    text = (
        f"Sentence: {row['Sentence']}, Language: {row['language']}\n"
        f"What is the emotion expressed in this sentence?\n"
        f"Only respond with one word, choosing exactly one of [disgust, anger, sad, happy, fear, surprise].\n"
        f"Answer in lowercase letters only:"
        )

    inputs = tokenizer(text, return_tensors="pt").to(device)
    with autocast(dtype=torch.float16):
        outputs = model.generate(**inputs, max_new_tokens=20)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    emotion = decoded.split("Answer in lowercase letters only:")[-1].strip()

    pred_emotions.append(emotion.lower())



In [ ]:
# Prepare ground truth
y_true = val_data["emotion"].str.lower().tolist()

macro_f1 = f1_score(y_true, pred_emotions, average="macro")
print("Macro F1 on validation set:", macro_f1)

print(classification_report(y_true, pred_emotions, digits=4))


In [ ]:
# val_ds[110]

In [ ]:
# from torch.cuda.amp import autocast

# text = (
#         f"Sentence: {val_ds[110]['Sentence']}, Language: {val_ds[110]['language']}\n"
#         f"What is the emotion expressed in this sentence?\n"
#         f"Only respond with one word, choosing exactly one of [disgust, anger, sad, happy, fear, surprise].\n"
#         f"Answer in lowercase letters only:"
#     )

# inputs = tokenizer(text, return_tensors="pt").to(device)

# with autocast(dtype=torch.float16):
#     outputs = model.generate(**inputs, max_new_tokens=20)

# print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer in lowercase letters only:")[-1])

# Prediction on Test Dataset

In [ ]:
from tqdm import tqdm
from torch.cuda.amp import autocast
device = "cuda:0"
model = model.to(device)

pred_emotions = []


for i, row in tqdm(test_data.iterrows(), total=len(test_data)):
    text = (
        f"Sentence: {row['Sentence']}, Language: {row['language']}\n"
        f"What is the emotion expressed in this sentence?\n"
        f"Only respond with one word, choosing exactly one of [disgust, anger, sad, happy, fear, surprise].\n"
        f"Answer in lowercase letters only:"
        )

    inputs = tokenizer(text, return_tensors="pt").to(device)
    with autocast(dtype=torch.float16):
        outputs = model.generate(**inputs, max_new_tokens=20)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    emotion = decoded.split("Answer in lowercase letters only:")[-1].strip()

    pred_emotions.append(emotion.lower())


In [ ]:
set(pred_emotions)

# Saving the prediction in submission.csv file

In [ ]:
submission = pd.DataFrame({
    "id": test_data['id'],
    "emotion": pred_emotions
})

submission.to_csv("submission.csv", index=False)
print("Submission saved!")

In [ ]:
pd.read_csv('/kaggle/working/submission.csv')

# Save the Model for further use

In [ ]:
save_path = "./my_gemma_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
import shutil

shutil.make_archive("my_gemma_model", "zip", save_path)